# task_04 Solution

In [1]:

from pathlib import Path
import pandas as pd

base = Path("../../tasks/task_04/data/")


In [2]:
batches = pd.read_csv(base / "batches.csv")
maintenance = pd.read_csv(base / "maintenance.csv")
qc = pd.read_csv(base / "qc.csv")
machines = pd.read_csv(base / "machines.csv")

Q1: Considering only batches produced within a week after the most recent maintenance on the same machine, which machine_id has the highest defect rate?

In [3]:
batches["batch_date"] = pd.to_datetime(batches["batch_date"])
maintenance["maintenance_date"] = pd.to_datetime(maintenance["maintenance_date"])

merged = batches.merge(qc, on="batch_id")

rows = []
for row in merged.itertuples(index=False):
    mm = maintenance[(maintenance["machine_id"] == row.machine_id) & (maintenance["maintenance_date"] <= row.batch_date)]
    if mm.empty:
        continue
    recent = mm["maintenance_date"].max()
    if (row.batch_date - recent).days <= 7:
        rows.append((row.machine_id, row.defect_units, row.units_produced))
post_maint = pd.DataFrame(rows, columns=["machine_id", "defect_units", "units_produced"])
defect_rate = post_maint.groupby("machine_id")["defect_units"].sum() / post_maint.groupby("machine_id")["units_produced"].sum()
q1 = str(defect_rate.sort_values(ascending=False).index[0])
print(q1)


M01


Q2: What is the scrap-adjusted yield for line B expressed as a percentage rounded to 2 decimals?

In [4]:
merged = batches.merge(qc, on="batch_id")
line_b = merged.merge(machines[["machine_id", "line"]], on="machine_id")
line_b = line_b[line_b["line"] == "B"]
scrap_adj = (line_b["units_produced"] - line_b["defect_units"] - line_b["rework_units"]).sum() / line_b["units_produced"].sum()
q2 = f"{(scrap_adj * 100):.2f}%"
print(q2)

94.68%
